In [1]:
import pandas as pd
import numpy as np
import joblib
import json

from xgboost import XGBRegressor

print("Libraries imported")

Libraries imported


In [3]:
df = pd.read_csv("data/processed/featured_train.csv")
print(df.shape)
df.head()

(844338, 26)


,Store,DayOfWeek,Promo,SchoolHoliday,CompetitionDistance,Promo2,Year,Month,Day,WeekOfYear,...,Sales,StateHoliday_0,StateHoliday_a,StateHoliday_b,StateHoliday_c,StoreType_b,StoreType_c,StoreType_d,Assortment_b,Assortment_c
0,1,5,1,1,1270.0,0,2015,7,31,31,...,5263,False,False,False,False,False,True,False,False,False
1,2,5,1,1,570.0,1,2015,7,31,31,...,6064,False,False,False,False,False,False,False,False,False
2,3,5,1,1,14130.0,1,2015,7,31,31,...,8314,False,False,False,False,False,False,False,False,False
3,4,5,1,1,620.0,0,2015,7,31,31,...,13995,False,False,False,False,False,True,False,False,True
4,5,5,1,1,29910.0,0,2015,7,31,31,...,4822,False,False,False,False,False,False,False,False,False


In [4]:
df = df.sort_values(by=["Year", "Month", "Day"]).reset_index(drop=True)
print(df.head())

   Store  DayOfWeek  Promo  SchoolHoliday  CompetitionDistance  Promo2  Year  \
0     85          2      0              1               1870.0       0  2013   
1    259          2      0              1                210.0       0  2013   
2    262          2      0              1               1180.0       0  2013   
3    274          2      0              1               3640.0       1  2013   
4    335          2      0              1                 90.0       1  2013   

   Month  Day  WeekOfYear  ...  Sales  StateHoliday_0  StateHoliday_a  \
0      1    1           1  ...   4220           False            True   
1      1    1           1  ...   6851           False            True   
2      1    1           1  ...  17267           False            True   
3      1    1           1  ...   3102           False            True   
4      1    1           1  ...   2401           False            True   

   StateHoliday_b  StateHoliday_c  StoreType_b  StoreType_c  StoreType_d  \
0   

In [5]:
X = df.drop("Sales", axis=1)
y = df["Sales"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (844338, 25)
y shape: (844338,)


log transform target

In [6]:
y_log = np.log1p(y)

In [7]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y_log.iloc[:split_index]
y_test = y_log.iloc[split_index:]

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (675470, 25) (675470,)
Test shape: (168868, 25) (168868,)


In [8]:
model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=10,
    min_child_weight=3,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print(model)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=3, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1200,
             n_jobs=-1, num_parallel_tree=None, ...)


In [9]:
model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100
)

print("Model trained successfully")

[0]	validation_0-rmse:0.42294	validation_1-rmse:0.40897
[100]	validation_0-rmse:0.27052	validation_1-rmse:0.28771
[200]	validation_0-rmse:0.19820	validation_1-rmse:0.23074
[300]	validation_0-rmse:0.15868	validation_1-rmse:0.19917
[400]	validation_0-rmse:0.13148	validation_1-rmse:0.17713
[500]	validation_0-rmse:0.11824	validation_1-rmse:0.16747
[600]	validation_0-rmse:0.10904	validation_1-rmse:0.16076
[700]	validation_0-rmse:0.10163	validation_1-rmse:0.15542
[800]	validation_0-rmse:0.09626	validation_1-rmse:0.15184
[900]	validation_0-rmse:0.09253	validation_1-rmse:0.14984
[1000]	validation_0-rmse:0.08977	validation_1-rmse:0.14894
[1100]	validation_0-rmse:0.08749	validation_1-rmse:0.14804
[1199]	validation_0-rmse:0.08549	validation_1-rmse:0.14742
Model trained successfully


In [11]:
joblib.dump(model, "artifacts/model.pkl")
joblib.dump(list(X.columns), "artifacts/columns.pkl")

print("Model saved to artifacts/model.pkl")
print("Columns saved to /artifacts/columns.pkl")

Model saved to artifacts/model.pkl
Columns saved to /artifacts/columns.pkl
